# **Project Name**    -Zomato Restaurant Analysis and Segmentation



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Nikhil
##### **Team Member 1 -**
##### **Team Member 2 -**
##### **Team Member 3 -**
##### **Team Member 4 -**

# **Project Summary -**

The project, Zomato Restaurant Analysis and Segmentation, aims to analyze restaurant performance, customer reviews, pricing, ratings, and cuisine preferences to identify meaningful restaurant segments and generate actionable business insights.

Data Preprocessing: The project uses two datasets: a restaurant metadata dataset containing 105 records and 6 features, and a customer review dataset containing 10,000 records and 7 features. Data preprocessing involved examining missing values, removing duplicate records, and converting columns into appropriate data types. Incomplete review records were removed, and the Collections column was excluded due to substantial missing values. Customer reviews were cleaned using lowercasing, URL and punctuation removal, stopword removal, and lemmatization, while important negation words were retained to preserve sentiment information.

Exploratory Data Analysis (EDA): Statistical summaries and visualizations were used to examine restaurant costs, ratings, and customer preferences. The analysis showed that approximately 72% of restaurants belong to the medium-price category, 18% to the low-price category, and 10% to the high-price category. A weak positive relationship was observed between restaurant cost and average rating, indicating that higher prices do not necessarily guarantee greater customer satisfaction. Approximately 70% of restaurants were classified as Good, 18% as Excellent, and 12% as Bad based on their average ratings.

Sentiment Analysis: Customer reviews were classified into Positive, Negative, and Neutral categories using Natural Language Processing (NLP). TF-IDF vectorization was used to convert cleaned review text into numerical features, and Logistic Regression was trained on manually labelled reviews. An 80:20 stratified train-test split was used to preserve sentiment proportions. The model was evaluated using accuracy, precision, recall, F1-score, and multiclass ROC-AUC. The trained model was subsequently used to predict sentiment for unlabelled reviews.

Restaurant Segmentation: K-Means and hierarchical clustering were applied using restaurant cost, ratings, price categories, cuisine features, and sentiment proportions. The Elbow Method, Silhouette Score, dendrogram analysis, Davies-Bouldin Index, and Calinski-Harabasz Score were used to evaluate clustering solutions. Based on the combined findings, five clusters were selected for the final segmentation.

The clusters revealed distinct restaurant profiles. Cluster 0 represents budget-friendly restaurants with an average rating of approximately 3.6. Cluster 1 contains the most expensive restaurants, with an average cost of approximately ₹1,910. Cluster 2 represents mid-range restaurants with an average cost of approximately ₹920 and a rating of around 3.5. Cluster 3 has the lowest average rating, approximately 2.8, highlighting opportunities to improve customer satisfaction. Cluster 4 has the highest average rating, approximately 4.4, despite costing less on average than Cluster 1.

Overall, the project integrates EDA, NLP, sentiment classification, and unsupervised machine learning to understand restaurant performance. The findings can help businesses improve pricing strategies, customer experience, competitive positioning, and targeted marketing decisions.

# **GitHub Link -**

https://github.com/nikhil1937/Zomato-Restaurant-Analysis-and-Segmentation/blob/main/Final_Zomato_Restaurant_Template.ipynb

# **Problem Statement**


To segment restaurants based on cost, ratings, and other relevant attributes using clustering techniques to identify meaningful groups and support data-driven business decisions.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd
import re
import ast

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text preprocessing
!pip install emoji
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine learning: preprocessing and feature extraction
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Machine learning: clustering
from sklearn.cluster import KMeans, DBSCAN, MeanShift, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Machine learning: train-test split and hyperparameter tuning
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

# Machine learning: classification
from sklearn.linear_model import LogisticRegression

# Model evaluation
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score,
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score
)

# Google Colab: Google Drive access
from google.colab import drive

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

### Dataset Loading

In [ ]:
# Mount Google Drive to access datasets and project files stored in Drive
drive.mount('/content/drive')

In [ ]:
# Load the Zomato restaurant names and metadata dataset
restaurantdata_df = pd.read_csv("/content/drive/MyDrive/Zomato Restaurant names and Metadata.csv")

# Load the Zomato restaurant reviews dataset
reviews_df = pd.read_csv("/content/drive/MyDrive/Zomato Restaurant reviews.csv")

### Dataset First View

In [ ]:
# Dataset First Look
restaurantdata_df.sample(10)

In [ ]:
# Dataset First Look
reviews_df.sample(10)

### Dataset Rows & Columns count

In [ ]:
# Display the total number of rows and columns in the dataset
print(f"Number of Rows    : {restaurantdata_df.shape[0]}")
print(f"Number of Columns : {restaurantdata_df.shape[1]}")

In [ ]:
# Display the total number of rows and columns in the dataset
print(f"Number of Rows    : {reviews_df.shape[0]}")
print(f"Number of Columns : {reviews_df.shape[1]}")

### Dataset Information

In [ ]:
# Create a copy of the dataset
restaurantdata_df_copy = restaurantdata_df.copy()
reviews_df_copy = reviews_df.copy()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_count = restaurantdata_df_copy.duplicated().sum()
print(f"Number of Duplicate Rows: {duplicate_count}")

In [ ]:
# Dataset Duplicate Value Count
duplicate_count = reviews_df_copy.duplicated().sum()
print(f"Number of Duplicate Rows: {duplicate_count}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("Missing values in Restaurant Dataset:\n")
print(restaurantdata_df_copy.isnull().sum())

print("\nMissing values in Reviews Dataset:\n")
print(reviews_df_copy.isnull().sum())

In [ ]:
# Missing values percentage in Metadata Dataset
print("Missing values percentage in Restaurant Dataset:\n")
print((restaurantdata_df_copy.isnull().sum() / len(restaurantdata_df_copy) * 100).round(2))

# Missing values percentage in Reviews Dataset
print("\nMissing values percentage in Reviews Dataset:\n")
print((reviews_df_copy.isnull().sum() / len(reviews_df_copy) * 100).round(2))

Checking column data types

In [ ]:
# Checking the data types of all columns in the metadata data frame.
restaurantdata_df_copy.dtypes

In [ ]:
# Checking the data types of all columns in the reviews data frame.
reviews_df_copy.dtypes

Missing values function

In [ ]:
def visualize_missing_values(df, dataset_name):

    # Calculate missing values
    missing_values = df.isnull().sum()

    # Create figure with 2 plots
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Bar Plot

    sns.barplot(
        x=missing_values.index,
        y=missing_values.values,
        ax=axes[0]
    )

    axes[0].set_title(
        f'Bar Plot of Missing Values - {dataset_name}'
    )
    axes[0].set_xlabel('Columns')
    axes[0].set_ylabel('Number of Missing Values')

    # Add values above bars
    for i, value in enumerate(missing_values.values):
        axes[0].text(
            i,
            value + 0.5,
            str(value),
            ha='center'
        )

    # Heatmap

    sns.heatmap(
        df.isnull(),
        cbar=True,
        yticklabels=False,
        ax=axes[1]
    )

    axes[1].set_title(
        f'Heatmap of Missing Values - {dataset_name}'
    )
    axes[1].set_xlabel('Columns')
    axes[1].set_ylabel('Rows')

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize missing values in the restaurant dataset
visualize_missing_values(
    restaurantdata_df_copy,
    "Restaurant Dataset"
)

In [ ]:
# Visualize missing values in the reviews dataset
visualize_missing_values(
    reviews_df_copy,
    "Reviews Dataset"
)

### What did you know about your dataset?

**Restaurant DataSet**
1. Restaurant Dataset has a total of 105 records and 6 features.
2. No duplicate records were found in the dataset.
3. The feature columns Timings and Collections have 1 and 54 missing values, respectively.
4. The Cost column was originally in object datatype and was converted to integer datatype.








**Review DataSet**
1. Review Dataset has a total of 10,000 records and 7 features.
2. There are 36 duplicate records in the dataset.
3. Feature columns Reviewer, Rating, Metadata, and Time have 38 missing values each, while Review has 45 missing values.
4. The Time column was originally in object datatype and was converted to datetime datatype.
5. The Rating column was originally in object datatype and was converted to numeric datatype.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Restaurant Dataset Columns:\n")
print(restaurantdata_df.columns.tolist())

print("\nReviews Dataset Columns:\n")
print(reviews_df.columns.tolist())

In [ ]:
# Dataset Describe
print("Restaurant Data Statistics:\n")
display(restaurantdata_df.describe(include='all').T)

In [ ]:
# Dataset Describe
print("Reviews Data Statistics:\n")
display(reviews_df.describe(include='all').T)


### Variables Description

**Restaurant Data**

Name: Name of the restaurant.

Links: Zomato URL/link of the restaurant.

Cost: Approximate cost for dining at the restaurant.

Collection: Zomato collection(s) associated with the restaurant.

Cuisines: Types of cuisines served by the restaurant

Timings: Operating hours of the restaurant.

**Review Data**

Restaurant: Name of the restaurant.

Reviewer: Name of the reviewer.

Review: Review written by the reviewer.

Rating: Rating given by the reviewer.

Metadata: Contains information about the reviewer's number of reviews and followers.

Time: Date and time when the review was written.

Pictures: Number of pictures uploaded by the reviewer with their review.




### Check Unique Values for each variable.

In [ ]:
# Check the number of unique values for each variable
print("Unique Values in Restaurant Dataset:")
print(restaurantdata_df_copy.nunique())

print("\nUnique Values in Reviews Dataset:")
print(reviews_df_copy.nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Converted 'Cost' column from object to integer data type.
restaurantdata_df_copy['Cost'] = (
    restaurantdata_df_copy['Cost']
    .str.replace(',', '')
    .astype(int)
)

In [ ]:
# Confirmed that the 'Cost' column has been converted to int64 data type.
restaurantdata_df_copy['Cost'].dtype

In [ ]:
# Checked the unique values in the 'Rating' column before data cleaning.
reviews_df_copy['Rating'].unique()

In [ ]:
# Replaced 'Like' with '5' and converted the 'Rating' column to numeric.
reviews_df_copy['Rating'] = reviews_df_copy['Rating'].replace('Like', '5')

reviews_df_copy['Rating'] = pd.to_numeric(
    reviews_df_copy['Rating'],
    errors='coerce'
)

In [ ]:
# Converted the 'Time' column to datetime data type.
reviews_df_copy['Time']=pd.to_datetime(reviews_df_copy['Time'])

In [ ]:
# Confirmed that the 'Time' column is already in datetime format.
reviews_df_copy['Time'].dtype

In [ ]:
# Remove the 'Collections' column from the Restaurant Dataset
restaurantdata_df_copy = restaurantdata_df_copy.drop(columns=['Collections'])

In [ ]:
# Remove incomplete review records because essential review information is missing
reviews_df_copy.dropna(
    subset=['Reviewer', 'Review', 'Rating', 'Metadata', 'Time'],
    inplace=True
)

In [ ]:
# Verify missing values after removal
print(reviews_df_copy.isnull().sum())

In [ ]:
# Remove duplicate records from the Reviews Dataset
reviews_df_copy.drop_duplicates(inplace=True)

In [ ]:
# Check the number of duplicate records after removal
print("Number of duplicate rows after removal:",
      reviews_df_copy.duplicated().sum())

### What all manipulations have you done and insights you found?

**Data Type Conversion:**

The Cost column in the Restaurant dataset was converted from object to integer (int) datatype.

The Rating column in the Reviews dataset was converted from object to numeric datatype.

The Time column in the Reviews dataset was converted from object to datetime datatype.

**Missing Value Treatment:**

A total of 45 rows (0.45% of the Reviews dataset) were removed because essential review-related information was missing. The missing records were concentrated in only two restaurants: American Wild Wings and Arena Eleven. Since the missing information included fields such as reviewer, review, rating, metadata, and time, meaningful imputation was not feasible for these records. As only 0.45% of the dataset was affected, removing these incomplete records minimized information loss while maintaining the quality and reliability of subsequent analysis.

The Collections column was removed from the Restaurant dataset because it contained 51.43% missing values, meaning more than half of the records did not have information available for this feature. Since a substantial proportion of the data was missing and the missing values could not be reliably imputed without potentially introducing misleading information, the column was excluded from further analysis to maintain data quality and reliability.

**Duplicate Row Removal:**

The Reviews dataset initially contained 36 duplicate records. These duplicate rows were removed to prevent repeated observations from affecting the accuracy and reliability of the analysis.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Extract Review/Post count

reviews_df_copy['Review/Post'] = (
    reviews_df_copy['Metadata']
    .str.extract(r'(\d+)\s*(?:Reviews?|Posts?)', expand=False)
    .astype('Int64')
)

reviews_df_copy['Followers'] = (
    reviews_df_copy['Metadata']
    .str.extract(r'(\d+)\s*Followers?', expand=False)
    .astype('Int64')
)

reviews_df_copy['Activity'] = (
    reviews_df_copy['Metadata']
    .str.extract(r'\d+\s*(Reviews?|Posts?)', expand=False)
    .str.lower()
    .str.rstrip('s')
)

# Check result
reviews_df_copy[
    ['Metadata', 'Review/Post', 'Followers', 'Activity']
].sample(30)

In [ ]:
# Convert the Cuisines column into a list of standardized cuisine names
restaurantdata_df_copy['Cuisine List'] = (
    restaurantdata_df_copy['Cuisines']
    .apply(lambda x: [c.strip().lower() for c in x.split(',')])
)

#Check the result
restaurantdata_df_copy[
    ['Cuisines', 'Cuisine List']
].sample(20)

In [ ]:
# Merge reviews with restaurant details using Restaurant and Name as matching columns
merged_df = reviews_df_copy.merge(
    restaurantdata_df_copy,
    left_on='Restaurant',
    right_on='Name',
)

In [ ]:
# Calculate the average rating for each restaurant
merged_df['Average Rating'] = (
    merged_df.groupby('Restaurant')['Rating']
    .transform('mean')
)

In [ ]:
# Get the top 15 highest-rated unique restaurants
n = 15

# Select Restaurant and Average Rating columns and remove duplicate rows
top_restaurants = merged_df[['Restaurant', 'Average Rating']].drop_duplicates()

# Select the top 15 restaurants based on Average Rating
top_restaurants = top_restaurants.nlargest(n, 'Average Rating')

# Display the result
print(top_restaurants)

In [ ]:
# Get the 15 lowest-rated unique restaurants
n = 15

# Select Restaurant and Average Rating columns and remove duplicate rows
lowest_restaurants = merged_df[['Restaurant', 'Average Rating']].drop_duplicates()

# Select the 15 restaurants with the lowest Average Rating
lowest_restaurants = lowest_restaurants.nsmallest(n, 'Average Rating')

# Display the result
print(lowest_restaurants)

In [ ]:
# Define a function to set the title and axis labels of a plot
def set_labels(ax, title, xlabel, ylabel):
    # Set the plot title
    ax.set_title(title)

    # Set the x-axis label
    ax.set_xlabel(xlabel)

    # Set the y-axis label
    ax.set_ylabel(ylabel)

In [ ]:
# Define a function to explode a column and remove duplicate rows
def explode_column(df, column_name):
    # Split list elements into separate rows
    df = df.explode(column_name)

    # Remove duplicates and reset the index
    df = df[[column_name]].drop_duplicates().reset_index(drop=True)

    # Return the processed DataFrame
    return df

In [ ]:
# Remove missing values from the Cost column
cost = merged_df['Cost'].dropna()

# Create a figure and axis for the plot
fig, ax = plt.subplots(figsize=(8, 5))

# Plot the cost distribution using a histogram with a KDE curve
sns.histplot(cost, bins=20, kde=True, stat='density', ax=ax)

# Set the plot title and axis labels
set_labels(ax, 'Distribution of Restaurant Cost', 'Cost', 'Density')

# Display the plot
plt.show()


**Observation:** The restaurant cost distribution is positively skewed (right-skewed), with most restaurants concentrated in the lower-to-middle cost range and a few restaurants having significantly higher costs. The distribution is not normal, and potential high-cost outliers are visible on the right tail.

**Business Insights:** Approximately 400–1,200 represents the main restaurant cost range, with the highest concentration around 500–750. Restaurants priced above 1,500 form a smaller segment, while a few restaurants reach 2,400–2,800. This suggests that the market is concentrated in the budget-to-mid-range segment, with a smaller premium segment offering opportunities for differentiated pricing and premium dining experiences.

In [ ]:
# Categorize restaurant costs into Low, Medium, and High price ranges
merged_df['Price_Range'] = merged_df['Cost'].apply(
    # Assign Low for costs below 500, Medium up to 1500, and High above 1500
    lambda x: 'Low' if x < 500
    else 'Medium' if x <= 1500
    else 'High'
)

In [ ]:
# Remove duplicate restaurants to keep only unique restaurants
unique_restaurants = merged_df.drop_duplicates(subset='Restaurant')

# Count the number of restaurants in each price range
price_counts = unique_restaurants['Price_Range'].value_counts()

# Create a figure for the pie chart
plt.figure(figsize=(7, 7))

# Plot the price range distribution with percentage labels and a 90-degree starting angle
plt.pie(price_counts, labels=price_counts.index, autopct='%1.1f%%', startangle=90)

# Set the chart title
plt.title('Distribution of Restaurants by Price Range')

# Display the pie chart
plt.show()

**Observation:** The distribution of restaurants by price range shows that 72% fall under the Medium category, 18% under the Low category, and 10% under the High category. The Medium category clearly dominates the market, while the High category represents the smallest proportion of restaurants.

**Business Insights:** The medium-price segment dominates with 72% of restaurants, indicating intense competition and the need for differentiation through food quality, service, and customer experience. The low-price segment (18%) can target price-sensitive customers through affordable meals and value-for-money offers. The high-price segment (10%) represents a smaller market where restaurants can focus on premium dining experiences, exclusive menus, and superior service to attract higher-spending customers.

In [ ]:
# Create a summary of unique restaurants with cost and average rating
restaurant_summary = (
    merged_df[['Restaurant', 'Cost', 'Average Rating']]
    # Remove duplicate rows and rows containing missing values
    .drop_duplicates()
    .dropna()
)

# Create a figure and axis for the scatter plot
fig, ax = plt.subplots(figsize=(8, 5))

# Plot restaurant cost against average rating
ax.scatter(
    restaurant_summary['Cost'],
    restaurant_summary['Average Rating']
)

# Set the plot title and axis labels
set_labels(
    ax,
    'Average Rating vs Cost',
    'Cost',
    'Average Rating'
)

# Display the plot
plt.show()

**Observation:** The scatter plot shows a weak positive relationship between restaurant cost and average rating, with considerable variation in ratings across all cost levels. Most restaurants are concentrated in the cost range of 400–1,000, with ratings primarily between 3.0 and 4.0. Some higher-cost restaurants achieve ratings above 4.5, but higher prices do not consistently guarantee higher ratings.

**Business Insights:** Restaurants should not assume that higher prices will automatically lead to better customer ratings. Since many restaurants priced between 400 and 1,000 have ratings around 3.0–4.0, businesses can gain a competitive advantage by improving food quality, service, and customer experience. Premium restaurants should focus on delivering value that justifies their higher prices, while mid-range restaurants can attract more customers by combining reasonable pricing with consistently high ratings.

In [ ]:
# Categorize restaurants based on their average ratings
merged_df['Restaurant Category'] = merged_df['Average Rating'].apply(
    # Assign Bad for ratings below 3, Good for ratings up to 4, and Excellent for ratings above 4
    lambda x: 'Bad' if x < 3
    else 'Good' if x <= 4
    else 'Excellent'
)

In [ ]:
# Remove duplicate restaurants to keep only unique restaurants
unique_restaurants = merged_df.drop_duplicates(subset='Restaurant')

# Count the number of restaurants in each category
category_counts = unique_restaurants['Restaurant Category'].value_counts()

# Create a figure and axis for the pie chart
fig, ax = plt.subplots(figsize=(7, 7))

# Plot the distribution of restaurant categories with percentage labels
ax.pie(
    category_counts.values,
    labels=category_counts.index,
    autopct='%1.1f%%',
    startangle=90
)

# Set the chart title
ax.set_title('Distribution of Restaurant Categories')

# Display the pie chart
plt.show()


**Observation:** The restaurant category distribution shows that 70% of restaurants are classified as Good, 18% as Excellent, and 12% as Bad. The Good category dominates, while the Excellent category represents a smaller proportion, indicating scope for more restaurants to improve their performance and move into the Excellent category.

**Business Insights:** Since 70% of restaurants are classified as Good, businesses should focus on improving food quality, service, and customer experience to move into the Excellent category, which currently accounts for 18%. The 12% classified as Bad represent an opportunity for improvement through customer feedback analysis, menu optimization, and better service consistency. New businesses can differentiate themselves by targeting unmet customer expectations and delivering consistently high-quality experiences.

In [ ]:
# Count the occurrences of each cuisine across unique restaurant-cuisine combinations
cuisine_counts = (
    merged_df[['Name', 'Cuisine List']]
    # Split each cuisine list into separate rows
    .explode('Cuisine List')
    # Remove duplicate restaurant-cuisine combinations
    .drop_duplicates()
    # Count the number of restaurants serving each cuisine
    ['Cuisine List']
    .value_counts()
)


In [ ]:
# Select the top 10 most common cuisines
top_cuisines = cuisine_counts.head(10)

# Create a figure for the bar chart
plt.figure(figsize=(10, 6))

# Plot the top cuisines
top_cuisines.plot(kind='bar')

# Set the x-axis label
plt.xlabel('Cuisine')

# Set the y-axis label
plt.ylabel('Number of Restaurants')

# Set the chart title
plt.title('Top 10 Most Popular Cuisines')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Display the chart
plt.show()


**Observation**: North Indian cuisine is the most popular, followed by Chinese and Continental cuisines. There is a significant gap between the top two cuisines and the remaining categories. The other cuisines show relatively similar representation, with Kebab being the least represented among the top 10. This indicates that restaurant offerings are concentrated around North Indian and Chinese cuisines.

**Business Insights:** North Indian and Chinese cuisines dominate the restaurant landscape, indicating strong market presence and potential business opportunities. Continental cuisine also shows notable representation, while cuisines such as Asian, Italian, and South Indian have comparatively fewer restaurants, potentially offering opportunities for market differentiation. Less-represented cuisines, such as Kebab, may provide niche opportunities; however, lower restaurant counts do not necessarily indicate higher demand or lower competition. Therefore, businesses should consider customer preferences, market demand, pricing, and existing competition before selecting a cuisine.

## **5.Sentiment Analysis**

In [ ]:
# Read the workbook containing the manually labelled reviews
manual_labels = pd.read_excel('/content/drive/MyDrive/Workbook.xlsx')

# Create a dictionary: Review → Sentiment
sentiment_map = dict(zip(manual_labels['Review'], manual_labels['Sentiment']))

# Add Sentiment column to merged_df
merged_df['Sentiment'] = merged_df['Review'].map(sentiment_map)

In [ ]:
# Define a function to convert emojis into their text descriptions
def convert_emojis(text):
    # Convert emojis into text and separate them with spaces
    return emoji.demojize(str(text), delimiters=(" ", " "))

# Convert emojis in reviews into text descriptions
merged_df['Review_Emoji_Text'] = merged_df['Review'].apply(convert_emojis)

In [ ]:
# Load English stopwords and convert them into a set
stop_words = set(stopwords.words('english'))

# Preserve important negation words for sentiment analysis
negation_words = {'not', 'no', 'never', 'nor'}
stop_words = stop_words - negation_words

# Initialize the WordNet lemmatizer
lemmatizer = WordNetLemmatizer()

# Define a function to preprocess review text
def preprocess_text(text):

    # Convert the input to a string
    text = str(text)

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize text into individual words
    words = text.split()

    # Remove stopwords and lemmatize remaining words
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    # Join processed words into a single string
    return ' '.join(words)


In [ ]:
# Apply text preprocessing to create a cleaned review column
merged_df['Clean_Review'] = merged_df['Review_Emoji_Text'].apply(preprocess_text)

In [ ]:
# Separate labelled reviews with available sentiment values
labelled_df = merged_df[merged_df['Sentiment'].notna()].copy()

# Separate unlabelled reviews with missing sentiment values
unlabelled_df = merged_df[merged_df['Sentiment'].isna()].copy()

In [ ]:
# Count labelled reviews before removing duplicates
before = len(merged_df[merged_df['Sentiment'].notna()])

# Remove duplicate reviews
labelled_df = merged_df[merged_df['Sentiment'].notna()].drop_duplicates(
    subset='Review', keep='first'
).reset_index(drop=True)

# Calculate duplicates removed
print("Labelled reviews before removing duplicates:", before)
print("Labelled reviews after removing duplicates:", len(labelled_df))
print("Duplicates removed:", before - len(labelled_df))

In [ ]:
# Select the cleaned review text as the input feature
X = labelled_df['Clean_Review']

# Select the sentiment labels as the target variable
y = labelled_df['Sentiment']

In [ ]:
# Count the number of reviews in each sentiment category
labelled_df['Sentiment'].value_counts()

In [ ]:

# Split the data into training and testing sets while preserving sentiment proportions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [ ]:
# Create a pipeline combining TF-IDF vectorization and Logistic Regression
pipeline = Pipeline([('tfidf', TfidfVectorizer()), ('model', LogisticRegression(max_iter=2000))])

# Define hyperparameters and their candidate values for tuning
param_grid = {'tfidf__max_features': [3000, 5000, 10000], 'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], 'model__C': [0.1, 1, 2, 5, 10], 'model__class_weight': [None, 'balanced']}

# Apply 5-fold stratified cross-validation to preserve class proportions in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform grid search using macro F1-score to select the best hyperparameter combination
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1)

# Train and evaluate each hyperparameter combination using cross-validation
grid_search.fit(X_train, y_train)

# Select the best-performing pipeline based on cross-validation macro F1-score
model = grid_search.best_estimator_

# Display the best hyperparameters and the corresponding cross-validation macro F1-score
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation F1:", grid_search.best_score_)

In [ ]:
# Retrieve the best model selected by GridSearchCV
best_model = grid_search.best_estimator_

In [ ]:
# Generate test predictions and predicted probabilities
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)

In [ ]:
# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Calculate multiclass ROC-AUC
print("Classes:", best_model.classes_)

roc_auc = roc_auc_score(
    y_test, y_prob, multi_class='ovr', average='weighted'
)

print("Multiclass ROC-AUC:", roc_auc)


In [ ]:
# Train final model using the best GridSearchCV pipeline
final_model = grid_search.best_estimator_
final_model.fit(X, y)

In [ ]:
# Separate positive and negative reviews
positive_reviews = labelled_df[labelled_df['Sentiment'] == 'Positive']['Clean_Review']
negative_reviews = labelled_df[labelled_df['Sentiment'] == 'Negative']['Clean_Review']

# Find the top 10 most frequent words in positive reviews
positive_words = pd.Series(' '.join(positive_reviews).split()).value_counts().head(10)

# Find the top 10 most frequent words in negative reviews
negative_words = pd.Series(' '.join(negative_reviews).split()).value_counts().head(10)

# Create side-by-side bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

positive_words.sort_values().plot(kind='barh', ax=axes[0])
axes[0].set_title('Top 10 Words in Positive Reviews')
axes[0].set_xlabel('Word Frequency')
axes[0].set_ylabel('Words')

negative_words.sort_values().plot(kind='barh', ax=axes[1])
axes[1].set_title('Top 10 Words in Negative Reviews')
axes[1].set_xlabel('Word Frequency')
axes[1].set_ylabel('Words')

plt.tight_layout()
plt.show()

**Observation:**

Positive reviews: Words such as good, great, and nice suggest customer satisfaction. Food, service, and ambience indicate important aspects of the restaurant experience.

Negative reviews: The word not is particularly important because it can reverse sentiment, as in “not good.” Ordered and experience may indicate complaints about orders or the overall dining experience.

Common words: food, good, place, service, chicken, and biryani appear in both graphs. Their meaning depends on the context in which they are used.

**Actionable Insights:**

Maintain food quality: Food appears frequently in both positive and negative reviews, indicating that it is an important aspect of the customer experience. Restaurants should maintain consistent food quality, taste, and freshness.

Improve service quality: Since service appears in both sentiment categories, restaurants should focus on faster service, staff behaviour, and order accuracy to improve customer satisfaction.

Enhance ambience: The presence of ambience in positive reviews suggests that a pleasant dining environment may contribute to positive customer experiences. Restaurants can maintain cleanliness, comfortable seating, and an appealing atmosphere.

Investigate negative feedback: The frequent occurrence of not in negative reviews highlights the importance of identifying phrases such as “not good” to understand customer dissatisfaction more accurately.

Monitor food-specific complaints: Words such as chicken, biryani, and ordered appear in negative reviews. Restaurants can investigate whether complaints relate to particular dishes, order accuracy, or food preparation.

Use customer feedback for continuous improvement: Restaurants should regularly analyse negative reviews to identify recurring issues and implement corrective actions.

In [ ]:
# Predict sentiments for unlabelled reviews
X_unlabelled = unlabelled_df['Clean_Review']

unlabelled_df['Sentiment'] = final_model.predict(X_unlabelled)

# Display sample predictions
print(unlabelled_df[['Review', 'Sentiment']].sample(10))

print("\nSentiment Distribution:")
print(unlabelled_df['Sentiment'].value_counts())

In [ ]:
# Combine labelled reviews and unlabelled reviews with predicted sentiments into one DataFrame
final_df = pd.concat(
    [labelled_df, unlabelled_df],
    ignore_index=True
)


In [ ]:
# Count the number of reviews in each sentiment category in the final dataset
final_df['Sentiment'].value_counts()

## **6.Clustering**

In [ ]:
# Create one row per restaurant
restaurant_features = final_df.groupby('Restaurant').agg({
    'Cost': 'first',
    'Average Rating': 'first',
    'Rating': 'mean',
    'Price_Range': 'first',
    'Restaurant Category': 'first',
    'Cuisine List': 'first'
}).reset_index()

# Calculate sentiment proportions
sentiment = pd.crosstab(final_df['Restaurant'], final_df['Sentiment'], normalize='index').reset_index()

# Merge sentiment proportions with restaurant features
restaurant_features = restaurant_features.merge(sentiment, on='Restaurant', how='left')

# Display the first five rows
display(restaurant_features.head())

In [ ]:
# Initialize the StandardScaler to standardize numerical features
scaler = StandardScaler()

# Standardize Cost and Average Rating and store the scaled values in new columns
restaurant_features[['cost_standardized', 'rating_standardized']] = scaler.fit_transform(
    restaurant_features[['Cost', 'Average Rating']]
)


In [ ]:
# Convert categorical columns into one-hot encoded numerical features
categorical_df = pd.get_dummies(restaurant_features[['Price_Range', 'Restaurant Category']], drop_first=False)

In [ ]:
# Convert string representations of cuisine lists into actual Python lists
restaurant_features['Cuisine List'] = restaurant_features['Cuisine List'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [ ]:
# Initialize MultiLabelBinarizer to encode multiple cuisines per restaurant
mlb = MultiLabelBinarizer()

# Convert cuisine lists into binary columns and create a DataFrame
cuisine_encoded = pd.DataFrame(mlb.fit_transform(restaurant_features['Cuisine List']), columns=mlb.classes_, index=restaurant_features.index)

In [ ]:
# Combine standardized numerical features, categorical features, and encoded cuisine features
clustering_df = pd.concat([restaurant_features[['cost_standardized', 'rating_standardized']], categorical_df, cuisine_encoded, restaurant_features[['Positive', 'Negative', 'Neutral']]], axis=1)

# Display a random sample of 5 rows from the clustering DataFrame
clustering_df.head(5)

In [ ]:
# Identify all categorical columns with Boolean data types
categorical_cols = clustering_df.select_dtypes(include='bool').columns

# Convert Boolean columns into integers (True = 1, False = 0)
clustering_df[categorical_cols] = clustering_df[categorical_cols].astype(int)

# Display a random sample of 5 rows
clustering_df.head(5)

In [ ]:
#Just for validation
pass
print("Shape:", clustering_df.shape)
print("Missing values:", clustering_df.isnull().sum().sum())
print("Duplicate rows:", clustering_df.duplicated().sum())
print("Non-numeric columns:", clustering_df.select_dtypes(exclude=np.number).columns.tolist())
print("Infinite values:", np.isinf(clustering_df.to_numpy(dtype=float)).sum())
print("Unique restaurants:", restaurant_features['Restaurant'].nunique())
print("Duplicate restaurant names:", restaurant_features['Restaurant'].duplicated().sum())

In [ ]:
# Count the total number of missing values in the clustering DataFrame
clustering_df.isnull().sum().sum()


In [ ]:

# Initialize lists to store inertia and silhouette scores for different cluster counts
inertia = []
silhouette_scores = []

# Evaluate K-Means clustering for cluster counts from 2 to 10
for k in range(2, 11):
    # Initialize the K-Means model with k clusters
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

    # Fit the model and assign cluster labels
    labels = kmeans.fit_predict(clustering_df)

    # Store the inertia value for the elbow method
    inertia.append(kmeans.inertia_)

    # Calculate and store the silhouette score
    silhouette_scores.append(silhouette_score(clustering_df, labels))


In [ ]:
for k in range(2, 11):
    print(
        f"K={k}, "
        f"Inertia={inertia[k-2]:.2f}, "
        f"Silhouette Score={silhouette_scores[k-2]:.4f}"
    )

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Method
ax[0].plot(range(2, 11), inertia, marker='o')
set_labels(ax[0], 'Elbow Method', 'Number of Clusters (K)', 'Inertia')
ax[0].set_xticks(range(2, 11))

# Silhouette Score
ax[1].plot(range(2, 11), silhouette_scores, marker='o')
set_labels(ax[1], 'Silhouette Score', 'Number of Clusters (K)', 'Silhouette Score')
ax[1].set_xticks(range(2, 11))

plt.tight_layout()
plt.show()

**Observation:** The Elbow Method shows a gradual decrease in inertia, with a possible elbow around \(K=4\) to \(K=5\). The Silhouette Score is highest at \(K=2\) (approximately 0.27), while \(K=5\) achieves the highest score among solutions with 3 or more clusters (approximately 0.187). Overall, the low silhouette scores suggest that the clusters have considerable overlap and are not well separated.

**Business Insight:** The Elbow Method suggests that 4–5 clusters may be suitable for restaurant segmentation. Although K=2 achieves the highest Silhouette Score, the score is relatively low, indicating limited cluster separation. K=5 has the second-highest Silhouette Score and aligns with the Elbow Method, making it a reasonable choice for further analysis. However, the final number of clusters will be determined after evaluating the results of other clustering algorithms to check whether they support a similar segmentation.

In [ ]:
# Check the shape of the clustering dataset
print("Shape of clustering_df:", clustering_df.shape)

# Check for missing values
print("Total missing values:", clustering_df.isnull().sum().sum())

# Display sample records
clustering_df.sample(5)

In [ ]:
# Create linkage matrix using Ward's method
linkage_matrix = linkage(clustering_df, method='ward')

In [ ]:
# Plot the truncated hierarchical clustering dendrogram
fig, ax = plt.subplots(figsize=(14, 7))

dendrogram(linkage_matrix, truncate_mode='lastp', p=20, leaf_rotation=45, leaf_font_size=10, ax=ax)

# Set the title and axis labels
set_labels(ax, 'Hierarchical Clustering Dendrogram', 'Cluster', 'Distance')

# Add a horizontal red dashed line at distance 60
ax.axhline(y=6, color='red', linestyle='--', linewidth=2)

# Display the dendrogram
plt.show()

**Observation:** The dendrogram shows that most clusters merge at distances below 6, while a few larger groups merge at higher distances, indicating differences in similarity among restaurant groups. At the distance threshold of 6, the horizontal cut intersects six clusters, with one pair of groups merging just above this threshold.

**Business Insight:** Although cutting the dendrogram at a distance of 6 produces 6 clusters, the final two groups merge at a distance only slightly above the cutoff. Therefore, considering the dendrogram structure and the similarity between these groups, K=5 can be considered a reasonable choice for hierarchical clustering.

**Decision:** The Elbow Method and Silhouette Score suggest that 4–5 clusters may be suitable for restaurant segmentation, while hierarchical clustering indicates a possible range of 5–6 clusters. In K-Means, K=5 achieves the second-highest Silhouette Score after K=2 and aligns with the Elbow Method. Hierarchical clustering also supports K=5, as the final two groups merge at a distance just above the selected cutoff. Considering the results of both methods, **K=5 is selected as the final number of clusters** for restaurant segmentation.

In [ ]:
# Initialize empty lists to store clustering evaluation scores
silhouette_scores_hc = []
davies_bouldin_scores_hc = []
calinski_harabasz_scores_hc = []

# Define the range of cluster numbers to evaluate (2 to 10)
cluster_range = range(2, 11)

# Evaluate hierarchical clustering for each cluster number
for k in cluster_range:
    # Generate cluster labels using the maximum distance criterion
    labels = fcluster(linkage_matrix, t=k, criterion='maxclust')

    # Calculate and store the Silhouette Score
    silhouette_scores_hc.append(silhouette_score(clustering_df, labels))

    # Calculate and store the Davies-Bouldin Score
    davies_bouldin_scores_hc.append(davies_bouldin_score(clustering_df, labels))

    # Calculate and store the Calinski-Harabasz Score
    calinski_harabasz_scores_hc.append(calinski_harabasz_score(clustering_df, labels))

In [ ]:
# Print the clustering evaluation scores for each number of clusters
for i, k in enumerate(cluster_range):
    print(
        f"K={k}, "
        f"Silhouette Score={silhouette_scores_hc[i]:.4f}, "
        f"Davies-Bouldin Index={davies_bouldin_scores_hc[i]:.4f}, "
        f"Calinski-Harabasz Score={calinski_harabasz_scores_hc[i]:.2f}"
    )

In [ ]:
# Apply hierarchical clustering to create 5 clusters
hierarchical = AgglomerativeClustering(n_clusters=5, linkage='ward')
restaurant_features['Hierarchical_Cluster'] = hierarchical.fit_predict(clustering_df)

# Map the cluster labels back to all rows in final_df
cluster_map = restaurant_features.set_index('Restaurant')['Hierarchical_Cluster']
final_df['Hierarchical_Cluster'] = final_df['Restaurant'].map(cluster_map)

# Display unique restaurants and their cluster labels
final_df[['Restaurant', 'Hierarchical_Cluster']].drop_duplicates().sort_values('Hierarchical_Cluster')


In [ ]:
# Calculate average rating and cost for each cluster
cluster_summary = restaurant_features.groupby('Hierarchical_Cluster').agg({'Average Rating': 'mean', 'Cost': 'mean'}).reset_index()

# Create a single graph with two bar plots
fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(cluster_summary))
width = 0.35

ax1.bar(x - width/2, cluster_summary['Cost'], width, color='steelblue', label='Average Cost')
ax1.set_xlabel('Cluster')
ax1.set_ylabel('Average Cost')
ax1.set_xticks(x)
ax1.set_xticklabels(cluster_summary['Hierarchical_Cluster'])

ax2 = ax1.twinx()
ax2.bar(x + width/2, cluster_summary['Average Rating'], width, color='orange', label='Average Rating')
ax2.set_ylabel('Average Rating')

ax1.set_title('Average Rating and Cost by Cluster')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
# Count the number of restaurants in each cluster
cluster_counts = restaurant_features['Hierarchical_Cluster'].value_counts().sort_index()

# Plot bar graph
cluster_counts.plot(kind='bar', figsize=(8, 5), color='steelblue')

plt.title('Number of Restaurants in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Restaurants')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Calculate average rating and cost for each cluster
cluster_summary = restaurant_features.groupby('Hierarchical_Cluster').agg({'Average Rating': 'mean', 'Cost': 'mean'}).reset_index()

# Create a bar graph
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(cluster_summary['Hierarchical_Cluster'].astype(str), cluster_summary['Average Rating'], color='steelblue')

# Set the title and axis labels
set_labels(ax, 'Average Rating by Cluster', 'Cluster', 'Average Rating')

# Display the graph
plt.tight_layout()
plt.show()

In [ ]:
# Calculate average rating and cost ratio for each cluster
cluster_summary = restaurant_features.groupby('Hierarchical_Cluster').agg({'Average Rating': 'mean', 'Cost': 'mean'}).reset_index()
cluster_summary['Rating_Cost_Ratio'] = cluster_summary['Average Rating'] / cluster_summary['Cost']

# Create a bar graph
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(cluster_summary['Hierarchical_Cluster'].astype(str), cluster_summary['Rating_Cost_Ratio'], color='steelblue')

# Set the title and axis labels
set_labels(ax, 'Average Ratings to Cost Ratio by Cluster', 'Cluster', 'Average Ratings to Cost Ratio')

# Display the graph
plt.tight_layout()
plt.show()

In [ ]:

# Clean cuisine names and split cuisines
cuisine_data = restaurant_features[['Hierarchical_Cluster', 'Cuisine List']].copy()
cuisine_data['Cuisine List'] = cuisine_data['Cuisine List'].astype(str).str.replace(r"[\[\]']", '', regex=True).str.split(',')
cuisine_data = cuisine_data.explode('Cuisine List')
cuisine_data['Cuisine List'] = cuisine_data['Cuisine List'].str.strip().str.lower()
cuisine_data = cuisine_data[cuisine_data['Cuisine List'] != '']

# Select top 15 cuisines
top_cuisines = cuisine_data['Cuisine List'].value_counts().head(15).index
cuisine_data = cuisine_data[cuisine_data['Cuisine List'].isin(top_cuisines)]

# Create heatmap
cuisine_cluster = cuisine_data.groupby(['Hierarchical_Cluster', 'Cuisine List']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 6))
sns.heatmap(cuisine_cluster, annot=True, fmt='d', cmap='YlGnBu', linewidths=0.5)

plt.title('Top 15 Cuisines Across Hierarchical Clusters')
plt.xlabel('Cuisine')
plt.ylabel('Cluster')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**1. Restaurant Segmentation**

Based on the Elbow Method, Silhouette Score, and hierarchical clustering dendrogram, five clusters were selected for the final segmentation. Although K=2 had the highest Silhouette Score of around 0.27, K=5 was considered more suitable for this analysis because it provided a more detailed classification of restaurants and was also supported by the Elbow Method and hierarchical clustering results. However, the relatively low Silhouette Scores suggest some overlap between the clusters, so the results should be viewed as broad restaurant categories rather than clearly separated groups.

**2. Key Business Insights from the Five Clusters**

Cluster 0 – Budget-Friendly Restaurants: These restaurants have the lowest average cost and an average rating of around 3.6. They cater to customers looking for affordable dining options. Maintaining food quality and service while keeping prices reasonable could help them retain customers.

Cluster 1 – Premium Restaurants: This cluster has the highest average cost, at approximately ₹1,910, but its average rating is lower than that of Cluster 4. This indicates that higher prices do not necessarily translate into greater customer satisfaction. These restaurants need to ensure that their food, service, and overall experience meet customer expectations.

Cluster 2 – Mid-Range Restaurants: With an average cost of around ₹920 and an average rating of 3.5, these restaurants fall into the middle of the market. Improvements in food quality, service, and the overall dining experience could help them attract more customers and improve their ratings.

Cluster 3 – Low-Rated Restaurants: This cluster has the lowest average rating, at approximately 2.8. These restaurants need to focus on understanding customer complaints and identifying common problems in their reviews. Addressing issues related to food quality, service, and pricing could help improve customer satisfaction.

Cluster 4 – High-Rated Restaurants: These restaurants have the highest average rating of around 4.4, despite having a lower average cost than Cluster 1. This suggests that customers may be more satisfied with the overall experience offered by these restaurants, even without paying premium prices.

**3. Cost Versus Customer Satisfaction**

The exploratory analysis showed a weak positive relationship between restaurant cost and average rating. However, the cluster results show that restaurants charging higher prices do not always receive better ratings. This suggests that pricing alone is not a reliable indicator of restaurant performance. Food quality, service, and the overall customer experience also need to be considered when evaluating customer satisfaction.

**4. Value-for-Money Analysis**

The ratings-to-cost ratio was used to compare customer ratings against the average cost of dining at a restaurant. A higher ratio indicates a better rating relative to the cost, making it useful for comparing restaurants from a value-for-money perspective. The cluster with the highest ratio can be considered the strongest performer on this measure. However, the actual ratio values should be compared before identifying the leading cluster.

**5. Sentiment Analysis and Business Applications**

Sentiment analysis was carried out using TF-IDF features and a Logistic Regression model to classify customer reviews as Positive, Negative, or Neutral. The resulting sentiment information was combined with restaurant-related variables, including cost, ratings, price categories, and cuisine information, for clustering.

This approach helped bring customer opinions into the segmentation process rather than relying only on numerical measures such as cost and ratings. The sentiment results can also help identify restaurants that receive consistently positive feedback and those that may need to address customer concerns.

**Overall Business Recommendation**

The findings suggest different priorities for each restaurant segment. Cluster 3 requires immediate attention because of its low average rating, while Cluster 4 can be studied to understand what contributes to its high customer satisfaction. Cluster 1 should focus on ensuring that its premium prices are justified by the experience it offers. Cluster 0 should continue to focus on affordability, and Cluster 2 can work on improving food quality and service to strengthen its position in the market.

Overall, the analysis shows that restaurant performance depends on more than just pricing. Combining cost, ratings, and customer sentiment provides a broader understanding of the market and can help restaurants identify areas for improvement.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***